In [64]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import pandas as pd
import random
import re

gsm8k = load_dataset("openai/gsm8k", "main", split="test")
arc = load_dataset("allenai/ai2_arc", "ARC-Challenge", split="test")

In [65]:
random.seed(42)
gsm_idx = random.sample(range(len(gsm8k)), 5)
arc_idx = random.sample(range(len(arc)), 5)
print(len(gsm8k), len(arc))

1319 1172


In [66]:
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-0.5B-Instruct")
model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-0.5B-Instruct", device_map="auto")
model.eval()

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 896)
    (layers): ModuleList(
      (0-23): 24 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=896, out_features=896, bias=True)
          (k_proj): Linear(in_features=896, out_features=128, bias=True)
          (v_proj): Linear(in_features=896, out_features=128, bias=True)
          (o_proj): Linear(in_features=896, out_features=896, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=896, out_features=4864, bias=False)
          (up_proj): Linear(in_features=896, out_features=4864, bias=False)
          (down_proj): Linear(in_features=4864, out_features=896, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((896,), eps=1e-06)
    (rotary_emb): Qwen2

In [67]:
def generate(user_prompt, max_new_tokens):
    messages = [{"role": "user", "content": user_prompt}]
    text = tokenizer.apply_chat_template(messages,tokenize=False,add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=max_new_tokens,do_sample=False, temperature=None, top_p=None, top_k=None)
    new_tokens = output[0, inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True), len(new_tokens)

In [68]:
import re
def gsm_prompt(q, mode):
    if mode == "direct":
        return f"{q}Reply with only the final number, nothing else. Provide only the answer."
    return f"{q}Solve step by step. At the end write 'Answer: <number>'."
    
def arc_prompt(q, options, mode):
    opts = "\n".join(f"{chr(65+i)}. {o}" for i, o in enumerate(options))
    base = f"{q}\n{opts}\n"
    if mode == "direct":
        return base + "Reply with only the letter of the correct option. Provide only the answer."
    return base + "Think step by step. At the end write 'Answer: <letter>'."


def extract_number(text):
    m = re.search(r"Answer:\s*\$?(-?[\d,]*\.?\d+)", text)
    if m:
        s = m.group(1)
    else:
        nums = re.findall(r"-?\d[\d,]*\.?\d*", text)
        s = nums[-1] if nums else None
    return s.replace(",", "").rstrip(".") if s else None
    
def extract_letter(text):
    m = re.search(r"Answer:\s*\(?([A-E])\b", text)
    if m:
        return m.group(1)
    m = re.search(r"\b([A-E])\b", text)
    return m.group(1) if m else None
    
def same_number(a, b):
    try:
        return float(a) == float(b)
    except (TypeError, ValueError):
        return False

In [79]:
rows = []
for i in gsm_idx:
    ex = gsm8k[i]
    gold = ex["answer"].split("####")[-1].strip().replace(",", "")
    for mode in ["direct", "cot"]:
        p = gsm_prompt(ex["question"], mode)
        txt, n = generate(p, 16 if mode == "direct" else 600)
        pred = extract_number(txt)
        rows.append(dict(question_id=f"gsm8k_{i}", dataset="gsm8k", model="Qwen/Qwen2.5-0.5B-Instruct", mode=mode,
                         prompt=p, generated_text=txt, extracted_answer=pred,
                         gold_answer=gold, correct=same_number(pred, gold), n_tokens=n))

for i in arc_idx:
    ex = arc[i]
    labels = ex["choices"]["label"]
    gold = chr(65 + labels.index(ex["answerKey"]))
    for mode in ["direct", "cot"]:
        p = arc_prompt(ex["question"], ex["choices"]["text"], mode)
        txt, n = generate(p, 16 if mode == "direct" else 600)
        pred = extract_letter(txt)
        rows.append(dict(question_id=f"arc_{i}", dataset="arc", model="Qwen/Qwen2.5-0.5B-Instruct", mode=mode,
                         prompt=p, generated_text=txt, extracted_answer=pred,
                         gold_answer=gold, correct=(pred == gold), n_tokens=n))

df = pd.DataFrame(rows)
df["truncated"] = (df["mode"] == "direct") & (df["n_tokens"] >= 16)|(df["mode"] == "cot") & (df["n_tokens"] >= 600)
df.to_csv("/kaggle/working/results_test10.csv", index=False)
df[["dataset", "mode", "extracted_answer", "gold_answer", "correct", "n_tokens","truncated"]]

,dataset,mode,extracted_answer,gold_answer,correct,n_tokens,truncated
0,gsm8k,direct,1550,2280,False,6,False
1,gsm8k,cot,1880,2280,False,293,False
2,gsm8k,direct,12,1,False,3,False
3,gsm8k,cot,60,1,False,462,False
4,gsm8k,direct,2,5,False,3,False
5,gsm8k,cot,0.8,5,False,245,False
6,gsm8k,direct,6,12,False,3,False
7,gsm8k,cot,6,12,False,310,False
8,gsm8k,direct,None,273,False,16,True
9,gsm8k,cot,1.5,273,False,436,False


In [80]:
for _, r in df.iterrows():
    print("="*60, r.dataset, r["mode"], "| верно:", r.correct)
    print(r.generated_text)

============================================================ gsm8k direct | верно: False
$1550
============================================================ gsm8k cot | верно: False
To solve this problem, we need to determine the amount of money raised by each girl and then sum these amounts.

1. **Kim's Amount**:
   - Kim raises $320 more than Alexandra.
   - Alexandra raises $430.
   - Therefore, Kim raises \( 430 + 320 = 750 \) dollars.

2. **Alexandra's Amount**:
   - Alexandra raises $430.

3. **Maryam's Amount**:
   - Maryam raises $400 more than Sarah.
   - Sarah raises $300.
   - Therefore, Maryam raises \( 300 + 400 = 700 \) dollars.

Now, let's add up the amounts raised by all three girls:

- Alexandra: $430
- Kim: $750
- Maryam: $700

Total amount raised = \( 430 + 750 + 700 \)

Let's calculate the total:

\[
430 + 750 = 1180
\]
\[
1180 + 700 = 1880
\]

So, the total amount raised by all three girls is $1880.

Therefore, the answer is:

\[
\boxed{1880}
\]
====================